# auto_packet_analyze_v3 — Kaggle 파이프라인

pcap 하나(또는 여러 개)를 넣으면 **extract_log → evidence → LLM 분석**까지 자동으로 돌린다.

## 실행 전 준비 (필수)
1. 우측 **Settings → Internet: ON** (apt/ollama/모델 다운로드에 필요)
2. 우측 **Settings → Accelerator: GPU** (gemma4:26b 는 GPU 필요)
3. **Add Input → 업로드한 pcap 데이터셋 연결** (pcap 은 `/kaggle/input/<dataset>/` 에 놓임)

## 실행 방법
- **최초 실행**: Run All. (setup + 모델 pull — 오래 걸림. 세션이 휘발성이라 새 세션마다 반복)
- **코드 수정 후 재실행**: 아래 **"4. ★ 재실행 시작점"** 셀부터 실행.
  git pull 만 하고 모델/패키지 재설치 없이 pcap 분석을 다시 돈다.
- extract/build_evidence 출력은 로그 파일로 숨기고 **run.py 출력만 표시**된다.

## 1. 설정 + 레포 clone (최초 1회)

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/DAADAISMYLIFE/auto_packet_analyze_v3.git"
WORK = "/kaggle/working"
REPO = f"{WORK}/auto_packet_analyze_v3"
MODEL = "gemma4:26b"          # setup.sh 의 모델 pull 용 (run.py 는 llm/config.py 의 MODEL 사용)
os.environ["MODEL"] = MODEL

if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
os.chdir(REPO)
print("repo :", REPO)
print("model:", MODEL)

## 2. 환경 구성 (최초 1회 — suricata/zeek/ollama 설치 + 모델 pull)
`setup.sh` 가 전부 처리한다. **오래 걸림.** 재실행 시에는 이 셀을 건너뛴다.

In [ ]:
# Kaggle 은 root 라 sudo 없이 설치됨. 모델 pull + test.py 스모크까지 수행.
# {REPO}/{MODEL} 는 Jupyter 가 위 셀의 파이썬 변수로 치환한다.
!cd {REPO} && MODEL={MODEL} bash setup.sh

## 3. 파이썬 패키지 (최초 1회)

In [ ]:
!pip install -q ollama   # run.py 가 쓰는 파이썬 패키지

## 4. ★ 재실행 시작점 — git pull + 전체 pcap 분석

코드를 고치고 다시 돌릴 때는 **이 셀부터** 실행하면 된다.
- `git pull` 로 코드만 최신화 (모델/패키지 재설치 없음 — 같은 세션이면 모델은 이미 pull 됨)
- ollama 서버가 죽어 있으면 다시 띄운다 (모델 다운로드는 다시 안 함)
- pcap 별로: extract_log + build_evidence 는 **로그 파일로 숨기고**, `run.py` 출력만 표시

In [ ]:
import os, glob, subprocess, time, urllib.request

# ── 셀 단독 실행에도 안전하도록 변수 재정의 (설치는 하지 않음) ──
WORK = "/kaggle/working"
REPO = f"{WORK}/auto_packet_analyze_v3"
MODEL = os.environ.get("MODEL", "gemma4:26b")
assert os.path.isdir(REPO), "레포가 없음 — 최초 실행이면 셀 1~3 부터 실행할 것"
os.chdir(REPO)

# ── 코드 최신화 (fast-forward 만; 모델/패키지 재설치 없음) ──
subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=False)

# ── ollama 서버 보장 (커널 재시작으로 죽었으면 재기동; 모델은 디스크에 있음) ──
def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=3)
        return True
    except Exception:
        return False

if not ollama_up():
    subprocess.Popen(["ollama", "serve"],
                     stdout=open(f"{WORK}/ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_up():
            break
        time.sleep(1)
print("ollama up:", ollama_up())

# ── 파이프라인: pcap 마다 extract → evidence (숨김) → run.py (표시) ──
pcaps = sorted(glob.glob("/kaggle/input/**/*.pcap", recursive=True)
               + glob.glob("/kaggle/input/**/*.pcapng", recursive=True))
print("발견된 pcap:", pcaps or "(없음 — Add Input 으로 데이터셋 연결했는지 확인)")

env = dict(os.environ, MODEL=MODEL)
for p in pcaps:
    name = os.path.splitext(os.path.basename(p))[0]
    print("\n" + "=" * 60 + f"\n  {name}\n" + "=" * 60)

    # 4-1) 추출 + evidence — 출력은 로그 파일로 (보고 싶으면 파일 열기)
    prep_log = f"{WORK}/prep_{name}.log"
    with open(prep_log, "w") as lf:
        subprocess.run(["bash", "scripts/extract_log.sh", p], cwd=REPO,
                       stdout=lf, stderr=subprocess.STDOUT, check=False)
        subprocess.run(["python", "scripts/build_evidence.py", name], cwd=REPO,
                       stdout=lf, stderr=subprocess.STDOUT, check=False)
    print(f"(준비 완료 — extract/evidence 로그: {prep_log})")

    # 4-2) LLM 분석 — run.py 출력만 이 셀에 표시 (llm/ 안에서 실행: from tools import ...)
    subprocess.run(["python", "run.py", name], cwd=os.path.join(REPO, "llm"),
                   env=env, check=False)

## 5. 보고서 확인
`run.py` 가 저장한 `reports/<name>.md` 를 렌더링해서 보여준다.

In [ ]:
import glob, os
from IPython.display import Markdown, display

reports = sorted(glob.glob(f"{REPO}/reports/*.md"))
if not reports:
    print("(보고서 없음 — 무혐의 판정이거나, run.py 출력은 위 파이프라인 셀 stdout 에서 확인)")
for rp in reports:
    display(Markdown(f"---\n# 📄 {os.path.basename(rp)}\n---"))
    display(Markdown(open(rp, encoding="utf-8").read()))